# Summary - Advanced Deep Learning

## Multi Layer Perceptron (MLP)

- ‘Simple’ Problems where the input is features
- Output layers in a CNN after feature extractions
- Feature transformation (for example after attention layers)
- Dimensionality reduction

Each node calculates its output $y$ based in the inputs $x$, the weights $w$ (on the edges), a bias value $b$ and the activation function $\sigma$:

$$
y = \sigma \left( \sum_{k=1}^{n} w_k x_k + b \right)
$$

:::{prf:theorem} Universal Approximation Theorem
There exists an activation function $\sigma$ which is analytic, strictly increasing and sigmoidal and has the following property: For any $f \in C[0,1]^d$ and $\varepsilon > 0$ there exist constants $d_i, c_{ij}, \theta_{ij}, \gamma_i$ and vectors $\mathbf{w}^{ij} \in \mathbb{R}^d$ for which

$$
\left| f(\mathbf{x}) - \sum_{i=1}^{6d+3} d_i \, \sigma \left( \sum_{j=1}^{3d} c_{ij} \sigma( \mathbf{w}^{ij} \cdot \mathbf{x} - \theta_{ij}) - \gamma_i \right) \right| < \varepsilon
$$

for all $\mathbf{x} = (x_1, \ldots, x_d) \in [0,1]^d$.

A simplified representation is

$$
f(\mathbf{x}) \approx \sum_{i=1}^{N(\varepsilon)} a_i \, \sigma(\mathbf{w}_i \cdot \mathbf{x} + b_i).
$$

:::

## CNN Architectures & Design Patterns

### Why CNNs?

In Deep Learning you usually start with raw data and you aim to learn features first - then based on these learned features, you would then try to solve the actual problem for which we can again use MLPs for example (i.e. in classification).

### Fundamental Design Principles
*   **Data Transformation:** Standard CNNs typically decrease spatial resolution (width/height) while increasing the number of channels (depth) to extract higher-level features.
*   **Downsampling:** Accomplished via **Pooling** (Max/Average) or **Strided Convolutions**. Strided convolutions (e.g., $stride=2$) are preferred in modern networks as the weights are learnable, allowing the network to adaptively reduce resolution.
*   **Normalization:** **Batch Normalization** is critical for training deep networks. it ensures data remains normalized (mean 0, variance 1) as it travels through layers, accelerating convergence and enabling higher learning rates.

:::{prf:definition} Convolution Output Size}
For an $n \times n$ image, $f \times f$ filter, padding $p$ and stride $s$, the convolution output size is:

$$
\left\lfloor \frac{n + 2p - f}{s} + 1 \right\rfloor
\times
\left\lfloor \frac{n + 2p - f}{s} + 1 \right\rfloor .
$$
:::

### Training Stability: Residual Connections
*   **Problem:** Vanishing/exploding gradients make training very deep networks (e.g., >30 layers) difficult; adding layers can actually degrade performance.
*   **Solution:** **Residual (Skip) Connections** add the original input $x$ to the output of a layer block $F(x)$, resulting in $y = F(x) + x$. This allows gradients to flow more easily through "shortcuts".

### Computational Efficiency: Separable Convolutions
*   **Standard Convolution:** A $7 \times 7$ filter has 50 parameters (49 weights + 1 bias).
*   **Separable Convolution:** Replaces a 2D filter with two 1D filters (e.g., $7 \times 1$ and $1 \times 7$), reducing parameters to 16 ((7*1 weights + 1 bias) * 2).
*   **Depth-wise Separable Convolution:** Used in MobileNet and Xception. It applies a single filter per input channel (**Depth-wise**) followed by a $1 \times 1$ convolution (**Point-wise**) to mix features across channels. This drastically reduces computation with minimal performance loss.

# Sequences, Attention & Transformers

### RNN Limitations
*   Recurrent Neural Networks (RNNs) process sequences step-by-step using a hidden state $H$.
*   **Vanishing Gradients:** Long sequences involve multiplying weights $W$ many times ($W^n$). If $W < 1$, gradients vanish; if $W > 1$, they explode. **LSTMs** and **GRUs** use "gates" to mitigate this and preserve long-term dependencies.

### Transformers

#### The Attention Mechanism

Attention allows a model to "look back" at all positions of an input sequence simultaneously, solving the bottleneck of single-vector encodings.
*   **The Trinity:** $Query$ (what I am looking for), $Key$ (what I have), and $Value$ (the information content).
*   **Calculation:** Calculated as a scaled dot-product:
    $$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
*   **Scaling:** The $\sqrt{d_k}$ factor keeps variance at 1, ensuring stable gradients regardless of the input dimension $d_k$.

#### Transformer Architecture

- Minimize computational complexity per layer
- Minimize path length between pair of words to facilitate learning of long-range dependencies
- Maximise the amount of computation that can be parallelized

![Transformer Architecture](Transformer-Architecture.png)

*   **Self-Attention:** $Q$, $K$, and $V$ all stem from the same input sequence via different learnable linear transforms.
*   **Multi-Head Attention:** Uses multiple sets of $(Q, K, V)$ to focus on different aspects of the sequence in parallel.
*   **Positional Encoding:** Since self-attention is permutation-invariant (order doesn't matter), unique sine/cosine vectors are added to input embeddings to inject sequence order.
*   **Vision Transformer (ViT):** Images are split into **patches** (e.g., $16 \times 16$), which are flattened and treated as tokens in a sequence for the transformer encoder.

## Reinforcement Learning (RL)

![Taxonomy of Reinforcement Learning Algorithms](RL-Taxonomy-Algos.png)

### Notation

- $A_t$ action at time $t$
- $S_t$ state at time $t$, typically due, stochastically, to $S_{t-1}$ and $A_{t-1}$
- $R_t$ reward at time $t$, typically due, stochastically, to $S_{t-1}$ and $A_{t-1}$
- $\pi$ policy (decision-making rule)
- $\pi(s)$ action taken in state $s$ under deterministic policy $\pi$
- $\pi(a \mid s)$ probability of taking action $a$ in state $s$ under stochastic policy $\pi$
- $G_t$ return following time $t$
- $v_\pi(s)$ value of state $s$ under policy $\pi$ (expected return)
- $v_*(s)$ value of state $s$ under the optimal policy
- $q_\pi(s, a)$ value of taking action $a$ in state $s$ under policy $\pi$
- $q_*(s, a)$ value of taking action $a$ in state $s$ under the optimal policy

### Markov Decision Processes (MDP)
*   **Markov Property:** The future depends only on the current state, not the history.
*   **Goal:** Maximize the **Return** ($G_t$), the cumulative (often discounted) reward:
    $$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots$$

:::{prf:definition} Dynamics of an MDP
$$
p(s', r \mid s, a) \doteq \Pr\{ S_t = s', R_t = r \mid S_{t-1} = s, A_{t-1} = a \}
$$

**Markov Property:** The future is independent of the past given the present.
:::

:::{prf:definition} Bellman Equation
$$
v_\pi(s) = \sum_a \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a)\,[r + \gamma v_\pi(s')], \quad \text{for all } s \in \mathcal{S}
$$

- $\pi(a \mid s)$ is the policy
- $p(s', r \mid s, a)$ is the MDP
- $r$ is the reward of the current state
- $\gamma v_\pi(s')$ is the discounted future reward
:::

### Key Algorithms
*   **Exploration vs. Exploitation:** $\epsilon$-greedy policy ensures the agent takes a random action with probability $\epsilon$ to explore the environment, rather than always being "greedy".
*   **Q-Learning:** An **off-policy** Temporal Difference (TD) learning method. It updates the $Q$-value of the current action based on the *maximum* possible $Q$-value of the next state:
    $$Q(s, a) \leftarrow Q(s, a) + \alpha [R + \gamma \max_{a'} Q(s', a') - Q(s, a)]$$
*   **SARSA:** An **on-policy** Temporal Difference (TD) method. It updates based on the action $a'$ actually taken by the current policy.
*   **PPO (Proximal Policy Optimization):** Uses a **clipped surrogate loss** to prevent the new policy from diverging too far from the old one, ensuring stable training.

## Explainable AI

### Local vs. Global Explanations
*   **Local:** Explains a specific instance (e.g., "Why was *this* loan denied?").
*   **Global:** Summarizes feature importance across the entire dataset.

### Saliency Mapping
*   **CAM (Class Activation Mapping):** Requires a specific architecture: **Global Average Pooling (GAP)** followed by a linear layer. The GAP combines spatial activations into a single scalar per feature map, which is then weighted to show importance.
*   **Grad-CAM:** More flexible; uses **gradients** of the output with respect to the final feature maps to calculate weights, allowing it to work with any CNN architecture.
*   **LIME:** Trains a **surrogate** linear model locally around a specific point. The **exponential kernel** in its loss function ensures that samples closer to the original input have much higher weight.

### Shapley Values (SHAP)
*   Based on game theory: how to fairly distribute the "payout" (prediction) among "players" (features).
*   **Axiom of Completeness:** The sum of all feature attributions must equal the model's prediction minus the baseline.
*   **Integrated Gradients:** A path-integral method that integrates gradients along a line from a baseline (e.g., black image) to the target image to solve the **saturation problem** (where gradients go to zero in trained models).

## Generative AI

### Explicit vs. Implicit Density
*   **Explicit:** Models the probability $P(x)$ directly (e.g., PixelRNN, VAE).
*   **Implicit:** Learns to sample from the distribution without explicitly calculating $P(x)$ (e.g., GANs).

### GANs (Generative Adversarial Networks)
*   **The Game:** A **Generator** creates fake images from noise, and a **Discriminator** tries to distinguish them from real data.
*   **Minmax Loss:**
    $$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$
*   **WGAN:** Uses the **Earth Mover (Wasserstein) distance**, which provides a stable gradient even if the generator and data distributions do not overlap.
*   **CycleGAN:** Learns style transfer between two domains without paired data (e.g., horses to zebras) using a **Cycle Consistency Loss** ($F(G(x)) \approx x$).

## Self-Supervised Learning (SSL)

### Pretext Tasks
The model solves "fake" tasks to learn a **backbone** representation.
*   Examples: Predicting image rotation, solving jigsaw puzzles, or colorization.
*   Success is measured by performance on **downstream tasks** (e.g., classification) using only a small amount of labeled data.

### Contrastive Learning (SimCLR)
*   **Method:** Create two augmented versions of the same image (positive pair) and maximize their similarity while minimizing similarity with all other images in the batch (negative pairs).
*   **Projection Head:** A small MLP that "absorbs" the contrastive loss, allowing the encoder to maintain general features.

### BYOL (Bootstrap Your Own Latent)
*   Learns representations **without negative examples**.
*   Uses two asymmetric networks (**Online** and **Target**). The Target weights are an **Exponential Moving Average (EMA)** of the Online weights, preventing the model from collapsing to a trivial constant output.